In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import glob, json, re, os
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

def norm_label(x):
    return 1 if str(x).strip().lower() in ("spam", "scam", "1", "suspicious") else 0

def load_any(path):
    df = pd.read_csv(path, encoding="latin-1")
    cols = {c.lower(): c for c in df.columns}
    if "v1" in cols and "v2" in cols:
        out = df[[cols["v1"], cols["v2"]]]
    else:
        t = next((cols[c] for c in ("msg", "message", "text", "sms", "body") if c in cols), None)
        l = next((cols[c] for c in ("label", "class", "category", "spam") if c in cols), None)
        if t is None or l is None:
            print("skip:", path, list(df.columns)); return None
        out = df[[l, t]]
    out.columns = ["label", "text"]
    print("loaded:", path, len(out))
    return out

files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
test_files = [f for f in files if "test40" in f.lower()]
train_files = [f for f in files if f not in test_files]

parts = [p for p in (load_any(f) for f in train_files) if p is not None]
df = pd.concat(parts, ignore_index=True).dropna()
df["y"] = df["label"].map(norm_label)
df["text"] = df["text"].astype(str).str.strip()
df = df.drop_duplicates(subset="text")
print("rows:", len(df), "| suspicious:", int(df.y.sum()), "| safe:", int((1 - df.y).sum()))

X_tr, X_te, y_tr, y_te = train_test_split(df.text, df.y, test_size=0.2, stratify=df.y, random_state=42)
vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b",
                      min_df=2, sublinear_tf=True)
clf = LogisticRegression(C=5, class_weight="balanced", max_iter=2000).fit(vec.fit_transform(X_tr), y_tr)

def report(name, texts, y):
    p = clf.predict(vec.transform(texts))
    print(f"\n== {name} ==")
    print(classification_report(y, p, target_names=["safe", "suspicious"], digits=3, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    print("False positive rate:", round(fp / max(fp + tn, 1), 4), "| missed scams:", fn)

report("Held-out split (optimistic)", X_te, y_te)
for f in test_files:
    t = pd.read_csv(f)
    report("test40 (REAL accuracy)", t.text, t.label.map(norm_label))

os.makedirs("/kaggle/working", exist_ok=True)
json.dump({
    "vocab": {k: int(v) for k, v in vec.vocabulary_.items()},
    "idf": vec.idf_.tolist(),
    "coef": clf.coef_[0].tolist(),
    "intercept": float(clf.intercept_[0]),
    "ngram": [1, 2], "token_pattern": r"\b\w+\b", "lowercase": True,
    "sublinear_tf": True, "l2_norm": True,
}, open("/kaggle/working/sms_model.json", "w"))

# JSON vs sklearn check (Kotlin ko yahi match karna hai)
m = json.load(open("/kaggle/working/sms_model.json"))
def score(text):
    toks = re.findall(r"\b\w+\b", text.lower())
    grams = toks + [f"{x} {y}" for x, y in zip(toks, toks[1:])]
    tf = {}
    for g in grams:
        if g in m["vocab"]: tf[g] = tf.get(g, 0) + 1
    w = {g: (1 + np.log(c)) * m["idf"][m["vocab"][g]] for g, c in tf.items()}
    n = np.sqrt(sum(v * v for v in w.values())) or 1.0
    z = sum(m["coef"][m["vocab"][g]] * v / n for g, v in w.items()) + m["intercept"]
    return 1 / (1 + np.exp(-z))
s = X_te.iloc[:200]
ref = clf.predict_proba(vec.transform(s))[:, 1]
print("\nJSON vs sklearn max diff:", float(np.abs(ref - np.array([score(x) for x in s])).max()))

loaded: /kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv 5572
loaded: /kaggle/input/datasets/junioralive/india-spam-sms-classification/spam_ham_india.csv 2267
rows: 7219 | suspicious: 1378 | safe: 5841

== Held-out split (optimistic) ==
              precision    recall  f1-score   support

        safe      0.988     0.997     0.992      1168
  suspicious      0.985     0.949     0.967       276

    accuracy                          0.988      1444
   macro avg      0.987     0.973     0.980      1444
weighted avg      0.988     0.988     0.987      1444

False positive rate: 0.0034 | missed scams: 14

JSON vs sklearn max diff: 2.220446049250313e-16


In [5]:
tests = [
 ("Your KYC has expired. Click bit.ly/kyc-upd now or account will be blocked", 1),
 ("Aapka SBI account block ho jayega, abhi KYC update karein: http://sbi-kyc.xyz", 1),
 ("Congratulations! You won Rs 25 lakh lottery. Call now to claim", 1),
 ("Aapko work from home job mili hai, Rs 5000 daily, WhatsApp karein 98XXXXXXXX", 1),
 ("Aapke paise galti se transfer ho gaye, refund ke liye link par click karein", 1),
 ("Customs fee pending for your parcel, pay Rs 99 at http://post-fee.top", 1),
 ("Rs 500 credited to your account via UPI from Rahul. Bal: Rs 2300", 0),
 ("Your OTP is 482913. Do not share it with anyone. -HDFC Bank", 0),
 ("Airtel: Recharge of Rs 299 successful. Validity 28 days", 0),
 ("Zomato: Your order is out for delivery, arriving in 10 mins", 0),
 ("Your IRCTC ticket PNR 4521367890 is confirmed. Train 12951", 0),
 ("Rs 1,200 debited from A/c XX1234 at Amazon on 22-Nov. -SBI", 0),
 ("Hey, kal 5 baje milte hain cafe mein?", 0),
 ("Your Amazon package has been delivered. Rate your experience", 0),
]
ok = 0
for t, y in tests:
    p = clf.predict_proba(vec.transform([t]))[0, 1]
    pred = int(p > 0.5); ok += (pred == y)
    print("OK " if pred == y else "XX ", round(p, 2), "| want", y, "|", t[:60])
print(f"\n{ok}/{len(tests)} sahi")

OK  0.99 | want 1 | Your KYC has expired. Click bit.ly/kyc-upd now or account wi
OK  0.51 | want 1 | Aapka SBI account block ho jayega, abhi KYC update karein: h
OK  0.99 | want 1 | Congratulations! You won Rs 25 lakh lottery. Call now to cla
XX  0.34 | want 1 | Aapko work from home job mili hai, Rs 5000 daily, WhatsApp k
OK  0.59 | want 1 | Aapke paise galti se transfer ho gaye, refund ke liye link p
OK  0.94 | want 1 | Customs fee pending for your parcel, pay Rs 99 at http://pos
XX  0.93 | want 0 | Rs 500 credited to your account via UPI from Rahul. Bal: Rs 
OK  0.37 | want 0 | Your OTP is 482913. Do not share it with anyone. -HDFC Bank
XX  0.86 | want 0 | Airtel: Recharge of Rs 299 successful. Validity 28 days
XX  0.85 | want 0 | Zomato: Your order is out for delivery, arriving in 10 mins
OK  0.23 | want 0 | Your IRCTC ticket PNR 4521367890 is confirmed. Train 12951
XX  0.94 | want 0 | Rs 1,200 debited from A/c XX1234 at Amazon on 22-Nov. -SBI
OK  0.09 | want 0 | Hey, kal 5 baje mil

In [6]:
tests = [
 ("Your KYC has expired. Click bit.ly/kyc-upd now or account will be blocked", 1),
 ("Aapka SBI account block ho jayega, abhi KYC update karein: http://sbi-kyc.xyz", 1),
 ("Congratulations! You won Rs 25 lakh lottery. Call now to claim", 1),
 ("Aapko work from home job mili hai, Rs 5000 daily, WhatsApp karein 98XXXXXXXX", 1),
 ("Aapke paise galti se transfer ho gaye, refund ke liye link par click karein", 1),
 ("Customs fee pending for your parcel, pay Rs 99 at http://post-fee.top", 1),
 ("Rs 500 credited to your account via UPI from Rahul. Bal: Rs 2300", 0),
 ("Your OTP is 482913. Do not share it with anyone. -HDFC Bank", 0),
 ("Airtel: Recharge of Rs 299 successful. Validity 28 days", 0),
 ("Zomato: Your order is out for delivery, arriving in 10 mins", 0),
 ("Your IRCTC ticket PNR 4521367890 is confirmed. Train 12951", 0),
 ("Rs 1,200 debited from A/c XX1234 at Amazon on 22-Nov. -SBI", 0),
 ("Hey, kal 5 baje milte hain cafe mein?", 0),
 ("Your Amazon package has been delivered. Rate your experience", 0),
]
ok = 0
for t, y in tests:
    p = clf.predict_proba(vec.transform([t]))[0, 1]
    pred = int(p > 0.5); ok += (pred == y)
    print("OK " if pred == y else "XX ", round(p, 2), "| want", y, "|", t[:60])
print(f"\n{ok}/{len(tests)} sahi")

OK  0.99 | want 1 | Your KYC has expired. Click bit.ly/kyc-upd now or account wi
OK  0.51 | want 1 | Aapka SBI account block ho jayega, abhi KYC update karein: h
OK  0.99 | want 1 | Congratulations! You won Rs 25 lakh lottery. Call now to cla
XX  0.34 | want 1 | Aapko work from home job mili hai, Rs 5000 daily, WhatsApp k
OK  0.59 | want 1 | Aapke paise galti se transfer ho gaye, refund ke liye link p
OK  0.94 | want 1 | Customs fee pending for your parcel, pay Rs 99 at http://pos
XX  0.93 | want 0 | Rs 500 credited to your account via UPI from Rahul. Bal: Rs 
OK  0.37 | want 0 | Your OTP is 482913. Do not share it with anyone. -HDFC Bank
XX  0.86 | want 0 | Airtel: Recharge of Rs 299 successful. Validity 28 days
XX  0.85 | want 0 | Zomato: Your order is out for delivery, arriving in 10 mins
OK  0.23 | want 0 | Your IRCTC ticket PNR 4521367890 is confirmed. Train 12951
XX  0.94 | want 0 | Rs 1,200 debited from A/c XX1234 at Amazon on 22-Nov. -SBI
OK  0.09 | want 0 | Hey, kal 5 baje mil

In [7]:
import glob, json, re, os
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

def norm_label(x):
    return 1 if str(x).strip().lower() in ("spam", "scam", "1", "suspicious") else 0

def load_any(path):
    df = pd.read_csv(path, encoding="latin-1")
    cols = {c.lower(): c for c in df.columns}
    if "v1" in cols and "v2" in cols:
        out = df[[cols["v1"], cols["v2"]]]
    else:
        t = next((cols[c] for c in ("msg", "message", "text", "sms", "body") if c in cols), None)
        l = next((cols[c] for c in ("label", "class", "category", "spam") if c in cols), None)
        if t is None or l is None:
            print("skip:", path, list(df.columns)); return None
        out = df[[l, t]]
    out = out.copy()
    out.columns = ["label", "text"]
    out["w"] = 4.0 if "my_hinglish" in path.lower() else 1.0
    print("loaded:", path, len(out), "| weight", out["w"].iloc[0])
    return out

files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("all csv files:", files)
test_files = [f for f in files if "test40" in f.lower()]
train_files = [f for f in files if f not in test_files]

parts = [p for p in (load_any(f) for f in train_files) if p is not None]
df = pd.concat(parts, ignore_index=True).dropna()
df["y"] = df["label"].map(norm_label)
df["text"] = df["text"].astype(str).str.strip()
df = df.drop_duplicates(subset="text")
print("rows:", len(df), "| suspicious:", int(df.y.sum()), "| safe:", int((1 - df.y).sum()))

X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    df.text, df.y, df.w, test_size=0.2, stratify=df.y, random_state=42)
vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b",
                      min_df=2, sublinear_tf=True)
clf = LogisticRegression(C=5, class_weight="balanced", max_iter=2000).fit(
    vec.fit_transform(X_tr), y_tr, sample_weight=w_tr)

def report(name, texts, y):
    p = clf.predict(vec.transform(texts))
    print(f"\n== {name} ==")
    print(classification_report(y, p, target_names=["safe", "suspicious"], digits=3, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    print("False positive rate:", round(fp / max(fp + tn, 1), 4), "| missed scams:", fn)

report("Held-out split (optimistic)", X_te, y_te)
for f in test_files:
    t = pd.read_csv(f)
    report("test40 (REAL accuracy)", t.text, t.label.map(norm_label))

os.makedirs("/kaggle/working", exist_ok=True)
json.dump({
    "vocab": {k: int(v) for k, v in vec.vocabulary_.items()},
    "idf": vec.idf_.tolist(),
    "coef": clf.coef_[0].tolist(),
    "intercept": float(clf.intercept_[0]),
    "ngram": [1, 2], "token_pattern": r"\b\w+\b", "lowercase": True,
    "sublinear_tf": True, "l2_norm": True,
}, open("/kaggle/working/sms_model.json", "w"))

m = json.load(open("/kaggle/working/sms_model.json"))
def score(text):
    toks = re.findall(r"\b\w+\b", text.lower())
    grams = toks + [f"{x} {y}" for x, y in zip(toks, toks[1:])]
    tf = {}
    for g in grams:
        if g in m["vocab"]: tf[g] = tf.get(g, 0) + 1
    w = {g: (1 + np.log(c)) * m["idf"][m["vocab"][g]] for g, c in tf.items()}
    n = np.sqrt(sum(v * v for v in w.values())) or 1.0
    z = sum(m["coef"][m["vocab"][g]] * v / n for g, v in w.items()) + m["intercept"]
    return 1 / (1 + np.exp(-z))
s = X_te.iloc[:200]
ref = clf.predict_proba(vec.transform(s))[:, 1]
print("\nJSON vs sklearn max diff:", float(np.abs(ref - np.array([score(x) for x in s])).max()))

all csv files: ['/kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv', '/kaggle/input/datasets/udgeethkaushik/sms-dataset-hinglish/my_hinglish.csv', '/kaggle/input/datasets/udgeethkaushik/sms-dataset-hinglish/test40.csv', '/kaggle/input/datasets/junioralive/india-spam-sms-classification/spam_ham_india.csv']
loaded: /kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv 5572 | weight 1.0
loaded: /kaggle/input/datasets/udgeethkaushik/sms-dataset-hinglish/my_hinglish.csv 480 | weight 4.0
loaded: /kaggle/input/datasets/junioralive/india-spam-sms-classification/spam_ham_india.csv 2267 | weight 1.0
rows: 7699 | suspicious: 1538 | safe: 6161

== Held-out split (optimistic) ==
              precision    recall  f1-score   support

        safe      0.991     0.994     0.993      1232
  suspicious      0.977     0.964     0.971       308

    accuracy                          0.988      1540
   macro avg      0.984     0.979     0.982      1540
weighted avg      0.988  